# 08 — Evaluation & Figures

Compiles cross-tier metrics from `data/artifacts/metrics_*.json` into a single comparison table and figure ready for the LaTeX Results section.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import ARTIFACTS_DIR, FIGURES_DIR, load_all_metrics
sns.set_theme(style="whitegrid")

In [ ]:
metrics = load_all_metrics()
print("Loaded metrics:", list(metrics.keys()))

In [ ]:
rows = []

# Lexicon
if "lexicon_vader" in metrics:
    m = metrics["lexicon_vader"]
    rows.append({"tier": "lexicon", "model": "VADER@0", "accuracy": m.get("accuracy")})

# Classical
if "classical_ml" in metrics:
    for r in metrics["classical_ml"]["results"]:
        rows.append({
            "tier": "classical",
            "model": r["model"],
            "accuracy": r.get("accuracy"),
            "f1_macro": r.get("f1_macro"),
            "roc_auc":  r.get("roc_auc"),
        })

# Neural
if "neural_distilbert" in metrics:
    m = metrics["neural_distilbert"]
    rows.append({
        "tier": "neural",
        "model": m.get("model"),
        "accuracy": m.get("eval_accuracy"),
        "f1_macro": m.get("eval_f1_macro"),
    })

results = pd.DataFrame(rows)
print(results.round(4))
results.to_csv(ARTIFACTS_DIR / "final_comparison.csv", index=False)

In [ ]:
# Bar chart across tiers
plot_df = results.dropna(subset=["accuracy"]).copy()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_df, x="model", y="accuracy", hue="tier", ax=ax, dodge=False)
ax.set_ylim(0.5, 1.0); ax.set_title("Accuracy by tier and model")
ax.set_xlabel(""); plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig_path = FIGURES_DIR / "08_tier_comparison.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path}")

In [ ]:
# Markdown summary block ready to paste into the LaTeX Results section
md_table = results.round(4).to_markdown(index=False)
out = (ARTIFACTS_DIR / "final_results_table.md")
out.write_text("## Final cross-tier results\n\n" + md_table + "\n")
print(out.read_text())

## What to copy into the LaTeX report

- `figures/08_tier_comparison.png` — Results section primary figure.
- `data/artifacts/final_comparison.csv` — Results table (convert to LaTeX with `pandas.to_latex` or paste the markdown).
- `figures/03_lexicon_signature_means.png`, `figures/04_classical_confusion.png`, `figures/06_zeroshot_desire_by_polarity.png` — supporting figures.
- `data/artifacts/zeroshot_examples.json` — qualitative examples for the Appendix.